<a href="https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My rule and its reason codes

I will use a simple refresh rule based on two observable signals only: staleness and visibility. The rule is meant for decision support, not for claiming a future outcome. I will not use trend labels, future windows, or any product flags as features.

- Signal B: a negative check on page-one CTR. I tested whether low CTR is concentrated on page-one pages; it was not, so I will not use that in the scoring rule.
- Signal A: staleness behind the refresh flag. Visible pages that have been untouched for a long time are the main candidates.

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    while True:
        if (current / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return current
        if current == current.parent:
            return Path.cwd().resolve()
        current = current.parent


repo_root = find_repo_root()
raw_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(raw_path)

for col in [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

visible = df["impressions_90d"] >= 500
stale = df["days_since_last_update"] >= 180
page_one = visible & (df["avg_position"] > 0) & (df["avg_position"] <= 10)
df["low_ctr_flag"] = (df["ctr"] < 0.5).astype(int)
df["signal1_bucket"] = np.where(stale & visible, "stale_visible", "other_visible")
df["signal2_bucket"] = np.where(page_one, "page_one", "deeper")

signal1_table = (
    df.loc[visible, ["signal1_bucket", "low_ctr_flag"]]
    .groupby("signal1_bucket")
    .agg(n=("low_ctr_flag", "size"), low_ctr_share=("low_ctr_flag", "mean"))
    .reset_index()
)

signal2_table = (
    df.loc[visible, ["signal2_bucket", "low_ctr_flag"]]
    .groupby("signal2_bucket")
    .agg(n=("low_ctr_flag", "size"), low_ctr_share=("low_ctr_flag", "mean"))
    .reset_index()
)

print("Signal 1: staleness behind refresh flags")
print(signal1_table.to_string(index=False))
print("\nVerdict: CONFIRMED")
print("Reason: stale and visible pages had a higher low-CTR share than other visible pages, although the stale group is small.")

print("\nSignal 2: CTR-vs-position behind CTR-fix logic")
print(signal2_table.to_string(index=False))
print("\nVerdict: OPPOSITE")
print("Reason: page-one pages did not show a higher low-CTR share than deeper pages, so I will not use page-one CTR in the rule.")

Signal 1: staleness behind refresh flags
signal1_bucket     n  low_ctr_share
 other_visible 16709       0.851517
 stale_visible    17       1.000000

Verdict: CONFIRMED
Reason: stale and visible pages had a higher low-CTR share than other visible pages, although the stale group is small.

Signal 2: CTR-vs-position behind CTR-fix logic
signal2_bucket    n  low_ctr_share
        deeper 9162       0.903296
      page_one 7564       0.789133

Verdict: OPPOSITE
Reason: page-one pages did not show a higher low-CTR share than deeper pages, so I will not use page-one CTR in the rule.


## 2. Build the ranked queue (writes the CSV)

The score is simple and transparent: pages that are both stale and visible get the strongest score; visible pages with high volume get a secondary score; everything else is left to monitor. The reason code is one label per row, and the action label tells a human what to do next.

In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    while True:
        if (current / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return current
        if current == current.parent:
            return Path.cwd().resolve()
        current = current.parent


repo_root = find_repo_root()
raw_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(raw_path)

for col in [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

visible = df["impressions_90d"] >= 500
stale = df["days_since_last_update"] >= 180
high_volume = df["impressions_90d"] >= 3000

df["score"] = np.where(stale & visible, 100, np.where(visible & high_volume, 65, 0)).astype(int)
df["reason_code"] = np.where(stale & visible, "stale_visible_page", np.where(visible & high_volume, "high_volume_visible_page", "monitor_only"))
df["action_label"] = np.where(stale & visible, "refresh", np.where(visible & high_volume, "review_for_refresh", "monitor"))

df = df.sort_values(["score", "impressions_90d", "days_since_last_update"], ascending=[False, False, True]).reset_index(drop=True)
df["rank"] = np.arange(1, len(df) + 1)

queue = df[["content_id", "client_id", "rank", "score", "reason_code", "action_label", "impressions_90d", "days_since_last_update", "avg_position", "ctr", "content_age_days"]].copy()

output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

metrics = {
    "rows": int(len(queue)),
    "top_score": float(queue["score"].max()),
    "median_score": float(queue["score"].median()),
    "rule": {
        "score_formula": "100 if stale_and_visible else 65 if visible_and_high_volume else 0",
        "reason_code": "stale_visible_page | high_volume_visible_page | monitor_only",
        "action_label": "refresh | review_for_refresh | monitor",
    },
    "output_path": str(output_path),
}
metrics_path = output_dir / "baseline_rule_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))

print(f"Wrote ranked queue to {output_path}")
print(f"Wrote metrics to {metrics_path}")
print(queue.head(10).to_string(index=False))

Wrote ranked queue to /content/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv
Wrote metrics to /content/flyrank-ml-internship-starter/work/outputs/baseline_rule_metrics.json
          content_id         client_id  rank  score        reason_code action_label  impressions_90d  days_since_last_update  avg_position  ctr  content_age_days
content_cf56e2e2e282 client_7f2253d7e2     1    100 stale_visible_page      refresh            61678                     194          19.7 0.15               231
content_7368877ea310 client_7f2253d7e2     2    100 stale_visible_page      refresh            59472                     194          24.8 0.13               231
content_1bfaa38ff26c client_7f2253d7e2     3    100 stale_visible_page      refresh            25715                     194          22.2 0.23               231
content_0a91db491d14 client_7f2253d7e2     4    100 stale_visible_page      refresh            13299                     193          10.5 0.49             

## 3. Top-20 review

For the top 20 rows, I note the action, the reason code, a confidence note, and what would make the ranking wrong. I am looking for evidence that the rule is overconfident or too broad.

In [4]:
queue = queue.copy()
queue["confidence_note"] = np.where(queue["ctr"] < 0.5, "medium", "low")

for _, row in queue.head(10).iterrows():
    if row["action_label"] == "refresh":
        why = f"stale and visible ({int(row['days_since_last_update'])} days since update, {int(row['impressions_90d'])} impressions)"
        wrong = "would be wrong if the page had been updated recently or the traffic spike was short-lived"
    else:
        why = f"high-volume visible page ({int(row['impressions_90d'])} impressions)"
        wrong = "would be wrong if the page's traffic fell below the visibility floor or the content had already been refreshed"
    print(f"{int(row['rank'])}. action={row['action_label']} | reason={row['reason_code']} | why={why} | confidence={row['confidence_note']} | {wrong}")

1. action=refresh | reason=stale_visible_page | why=stale and visible (194 days since update, 61678 impressions) | confidence=medium | would be wrong if the page had been updated recently or the traffic spike was short-lived
2. action=refresh | reason=stale_visible_page | why=stale and visible (194 days since update, 59472 impressions) | confidence=medium | would be wrong if the page had been updated recently or the traffic spike was short-lived
3. action=refresh | reason=stale_visible_page | why=stale and visible (194 days since update, 25715 impressions) | confidence=medium | would be wrong if the page had been updated recently or the traffic spike was short-lived
4. action=refresh | reason=stale_visible_page | why=stale and visible (193 days since update, 13299 impressions) | confidence=medium | would be wrong if the page had been updated recently or the traffic spike was short-lived
5. action=refresh | reason=stale_visible_page | why=stale and visible (194 days since update, 7812 i

## 4. Weak picks + leakage check

The weakest picks are the ones that look too easy or too broad. I also checked that the rule uses only observable signals from the current window and does not pull in product flags or future-window outcomes.

In [5]:
weak_picks = queue.head(10).copy()
weak_picks["weakness_note"] = np.where(
    weak_picks["action_label"] == "refresh",
    "This rule can over-rank very stale pages that are not actually worth a refresh if they are already low-volume or have recently changed.",
    "This rule can over-rank high-volume pages that are simply visible, not necessarily in need of action.",
)
print(weak_picks[["rank", "score", "action_label", "reason_code", "impressions_90d", "days_since_last_update", "ctr", "weakness_note"]].to_string(index=False))

leakage_check = {
    "uses_future_window": False,
    "uses_trend_label": False,
    "uses_product_flags": False,
    "features_used": ["impressions_90d", "days_since_last_update", "avg_position", "ctr"],
}
print("\nLeakage check")
print(json.dumps(leakage_check, indent=2))

 rank  score action_label        reason_code  impressions_90d  days_since_last_update  ctr                                                                                                                           weakness_note
    1    100      refresh stale_visible_page            61678                     194 0.15 This rule can over-rank very stale pages that are not actually worth a refresh if they are already low-volume or have recently changed.
    2    100      refresh stale_visible_page            59472                     194 0.13 This rule can over-rank very stale pages that are not actually worth a refresh if they are already low-volume or have recently changed.
    3    100      refresh stale_visible_page            25715                     194 0.23 This rule can over-rank very stale pages that are not actually worth a refresh if they are already low-volume or have recently changed.
    4    100      refresh stale_visible_page            13299                     193 0.49 T

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.